In [4]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
sg_api_key = os.getenv("SG")


In [9]:
from langchain_scrapegraph.tools import SmartScraperTool

tool = SmartScraperTool(api_key=sg_api_key)

In [10]:
model = "gemini-2.0-flash"

In [11]:
result = tool.invoke({
    "website_url": "https://www.amazon.com",
    "user_prompt": "Extract the main heading and first paragraph"
})

In [12]:
result

{'main_heading': 'Prime Video',
 'first_paragraph': 'No paragraph found for the main heading Prime Video'}

In [13]:
from typing import List
from pydantic import BaseModel, Field

class WebsiteInfo(BaseModel):
    title:str=Field(description="The main title of the web page")
    description:str=Field(description="The main description of the web page")
    urls:List[str] = Field(description="The urls inside the webpages")

In [15]:
tool = SmartScraperTool(llm_output_schema=WebsiteInfo , api_key=sg_api_key)

In [16]:
result = tool.invoke({
    "website_url": "https://www.amazone.com",
    "user_prompt": "Extract the website information"
})

result

{'title': 'Cookies and advertising choices',
 'description': 'If you agree, we may use your personal information from any of these Amazon services to personalize the ads we show you on other services. For example, we may use your Prime Video Watch history to personalize the ads we show you on our Stores or on Fire TV. We may also use personal information we receive from third parties (like demographic information).',
 'urls': ['https://www.amazone.com/-/en/gp/help/customer/display.html?nodeId=T1fdzp9ecINEaVWTyY',
  'https://www.amazone.com/-/en/gp/help/customer/display.html?nodeId=201890250&ref_=footer_cookies_notice',
  'https://www.amazone.com/-/en/privacyprefs/retail/partners',
  'https://www.amazone.com/-/en/privacyprefs/retail?oCT=ads&ref_=portal_banner_cpp',
  'https://www.amazone.com/-/en/gp/help/customer/display.html?nodeId=201909010&ref_=footer_privacy',
  'https://www.amazone.com/-/en/b/?_encoding=UTF8&node=1787884031&pd_rd_w=8Bcro&content-id=amzn1.sym.43e5654a-1e60-416e-9e6f

In [17]:
from langchain_scrapegraph.tools import SearchScraperTool

tool = SearchScraperTool(api_key=sg_api_key)

In [18]:
result = tool.invoke({
    "user_prompt": "Find the best restaurants in San Francisco",
})

In [19]:
result

{'top_rated_restaurants': [{'name': 'Sushi Hashiri',
   'cuisine': 'Sushi',
   'rating': 5.0},
  {'name': 'Dragon Horse', 'cuisine': 'Izakaya', 'rating': 5.0},
  {'name': 'Kokkari Estiatorio', 'cuisine': 'Mediterranean', 'rating': 4.9},
  {'name': 'Saison', 'cuisine': 'American', 'rating': 4.9},
  {'name': "Harris'", 'cuisine': 'Steakhouse', 'rating': 4.9},
  {'name': 'Frascati', 'cuisine': 'Italian', 'rating': 4.9},
  {'name': 'Sushi Hakko', 'cuisine': 'Japanese', 'rating': 4.9},
  {'name': 'Sushi Hon', 'cuisine': 'Sushi', 'rating': 4.9},
  {'name': 'MKT Restaurant and Bar', 'cuisine': 'Californian', 'rating': 4.9},
  {'name': 'House of Prime Rib', 'cuisine': 'Steakhouse', 'rating': 4.8}]}

In [33]:
from langchain.agents import initialize_agent, AgentType
from langchain_scrapegraph.tools import SmartScraperTool
from langchain_openai import ChatOpenAI

# Initialize tools
tools = [
    SmartScraperTool(api_key=sg_api_key),
]

# Create an agent
agent = initialize_agent(
    tools=tools,
    llm=ChatOpenAI(temperature=0),
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Use the agent
response = agent.run("""
    Visit example.com, make a summary of the content and extract the main heading and first paragraph
""")

ValueError: ZeroShotAgent does not support multi-input tool SmartScraper.

In [ ]:
from burr.core import action, State, ApplicationBuilder
from scrapegraph_py import Client
import lancedb
from lancedb.pydantic import LanceModel, Vector
import openai
import tiktoken
from typing import List, Optional

# Schema for storing text chunks
class TextDocument(LanceModel):
    url: str
    position: int
    text: str
    vector: Vector(dim=1536)  # OpenAI embedding dimensions

# Action to fetch and convert webpage to markdown
@action(reads=[], writes=["markdown_content"])
def fetch_webpage(state: State, webpage_url: str) -> State:
    client = Client()
    response = client.markdownify(website_url=webpage_url)
    return state.update(markdown_content=response["result"])

# Action to embed and store chunks
@action(reads=["markdown_content"], writes=[])
def embed_and_store(state: State, webpage_url: str) -> State:
    chunks = get_text_chunks(state["markdown_content"])
    con = lancedb.connect("./webpages")
    table = con.create_table("chunks", schema=TextDocument)
    table.add([{
        "text": chunk,
        "url": webpage_url,
        "position": i
    } for i, chunk in enumerate(chunks)])
    return state

# Action to answer questions
@action(reads=[], writes=["llm_answer"])
def ask_question(state: State, user_query: str) -> State:
    chunks_table = lancedb.connect("./webpages").open_table("chunks")
    relevant_chunks = chunks_table.search(user_query).limit(3).to_list()
    
    response = openai.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": f"Answer based on: {relevant_chunks}"},
            {"role": "user", "content": user_query}
        ]
    )
    return state.update(llm_answer=response.choices[0].message.content)